# Robustness Check: RESOLVE-IPD Reconstruction

In this notebook, we assess whether the downstream causal survival analysis is sensitive to the upstream reconstruction engine.

Previously, the real-data analysis used reconstructed individual patient data (IPD) obtained from KM-GPT. Here, we instead use IPD reconstructed via RESOLVE-IPD and repeat the downstream workflow.

The purpose is to compare:
- naive pooled survival effects
- target-population weighted effects
- Bayesian nonparametric uncertainty summaries

while keeping the auxiliary summaries and downstream modeling choices fixed.

*** Note ***

This robustness comparison is for overall arm-level trial comparison but not a fully matched subgroup robustness check because of the dataset mis-match (KMGPT was subgroup specific reconstruction, RESOLVEIPD was overall arm reconstruction). we are aiming to answer:
> If we replace KM-GPT with higher-fidelity reconstructed arm-level IPD, do the downstream target-weighted survival conclusions remain qualitatively similar?

In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from src.causal.estimands import pooled_survival_difference
from src.causal.pooling import (
    compute_covariate_means,
    reweight_to_target,
    weighted_survival_difference,
)
from src.covariates.augmentation import (
    simulate_covariates_from_aux,
    summarize_simulated_covariates,
)

np.random.seed(733)

DATA_DIR = Path("../data/processed")

# Load auxiliary summaries and KM-GPT benchmark results
combined_aux = pd.read_csv(DATA_DIR / "combined_auxiliary_summaries.csv")
naive_kmgpt = pd.read_csv(DATA_DIR / "naive_pooled_effect.csv")
weighted_kmgpt = pd.read_csv(DATA_DIR / "weighted_estimate.csv")
bnp_kmgpt = pd.read_csv(DATA_DIR / "bnp_summary.csv")

print("Loaded comparison benchmarks.")
print("combined_aux:", combined_aux.shape)
print("naive_kmgpt:", naive_kmgpt.shape)
print("weighted_kmgpt:", weighted_kmgpt.shape)
print("bnp_kmgpt:", bnp_kmgpt.shape)

Loaded comparison benchmarks.
combined_aux: (4, 7)
naive_kmgpt: (1, 4)
weighted_kmgpt: (1, 4)
bnp_kmgpt: (1, 4)


In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

DATA_DIR = Path("../data/raw/resolveipd_data")

# Load arm-level RESOLVE-IPD outputs
attr_chemo = pd.read_csv(DATA_DIR / "Attraction3 Chemo(in).csv")
attr_nivo = pd.read_csv(DATA_DIR / "Attraction3 Nivolumab(in).csv")

escort_chemo = pd.read_csv(DATA_DIR / "ESCORT Chemo(in).csv")
escort_cam = pd.read_csv(DATA_DIR / "ESCORT Camrelizumab(in).csv")

keynote_chemo = pd.read_csv(DATA_DIR / "Keynote181 Chemo(in).csv")
keynote_pembro = pd.read_csv(DATA_DIR / "Keynote181 Pembrolizumab(in).csv")

print("ATTRACTION-3 chemo:", attr_chemo.shape)
print("ATTRACTION-3 nivolumab:", attr_nivo.shape)
print("ESCORT chemo:", escort_chemo.shape)
print("ESCORT camrelizumab:", escort_cam.shape)
print("KEYNOTE-181 chemo:", keynote_chemo.shape)
print("KEYNOTE-181 pembrolizumab:", keynote_pembro.shape)

ATTRACTION-3 chemo: (209, 2)
ATTRACTION-3 nivolumab: (210, 2)
ESCORT chemo: (220, 2)
ESCORT camrelizumab: (228, 2)
KEYNOTE-181 chemo: (203, 2)
KEYNOTE-181 pembrolizumab: (198, 2)


In [2]:
def standardize_resolve_arm(df, trial_id, arm_label, treatment):
    out = df.copy()

    out = out.rename(columns={"event": "event", "time": "time"})
    out["trial_id"] = trial_id
    out["arm_label"] = arm_label
    out["treatment"] = treatment

    out = out[["trial_id", "time", "event", "arm_label", "treatment"]].copy()
    return out


# Standardize each arm
attr_chemo_std = standardize_resolve_arm(
    attr_chemo, trial_id="ATTRACTION-3", arm_label="Chemotherapy", treatment=0
)
attr_nivo_std = standardize_resolve_arm(
    attr_nivo, trial_id="ATTRACTION-3", arm_label="Nivolumab", treatment=1
)

escort_chemo_std = standardize_resolve_arm(
    escort_chemo, trial_id="ESCORT", arm_label="Chemotherapy", treatment=0
)
escort_cam_std = standardize_resolve_arm(
    escort_cam, trial_id="ESCORT", arm_label="Camrelizumab", treatment=1
)

keynote_chemo_std = standardize_resolve_arm(
    keynote_chemo, trial_id="KEYNOTE-181", arm_label="Chemotherapy", treatment=0
)
keynote_pembro_std = standardize_resolve_arm(
    keynote_pembro, trial_id="KEYNOTE-181", arm_label="Pembrolizumab", treatment=1
)

# Combine all trials
df_resolve = pd.concat(
    [
        attr_chemo_std, attr_nivo_std,
        escort_chemo_std, escort_cam_std,
        keynote_chemo_std, keynote_pembro_std,
    ],
    ignore_index=True,
)

print("Combined RESOLVE-IPD dataset shape:", df_resolve.shape)
df_resolve.head()

Combined RESOLVE-IPD dataset shape: (1268, 5)


,trial_id,time,event,arm_label,treatment
0,ATTRACTION-3,0.631283,1,Chemotherapy,0
1,ATTRACTION-3,0.797091,1,Chemotherapy,0
2,ATTRACTION-3,1.163915,1,Chemotherapy,0
3,ATTRACTION-3,1.344771,1,Chemotherapy,0
4,ATTRACTION-3,1.621011,1,Chemotherapy,0


In [3]:
resolve_summary = (
    df_resolve.groupby(["trial_id", "arm_label"])
    .agg(
        n=("time", "size"),
        events=("event", "sum"),
        event_rate=("event", "mean"),
        median_time=("time", "median"),
    )
    .reset_index()
)

resolve_summary

,trial_id,arm_label,n,events,event_rate,median_time
0,ATTRACTION-3,Chemotherapy,209,173,0.827751,8.021619
1,ATTRACTION-3,Nivolumab,210,160,0.761905,10.397406
2,ESCORT,Camrelizumab,228,172,0.754386,8.157870
3,ESCORT,Chemotherapy,220,191,0.868182,6.151607
4,KEYNOTE-181,Chemotherapy,203,182,0.896552,6.928100
5,KEYNOTE-181,Pembrolizumab,198,166,0.838384,8.227040


In [5]:
from pathlib import Path

output_dir = Path("../data/processed")
output_dir.mkdir(parents=True, exist_ok=True)

df_resolve.to_csv(output_dir / "resolve_ipd_harmonized.csv", index=False)
resolve_summary.to_csv(output_dir / "resolve_ipd_summary.csv", index=False)

print("Saved RESOLVE-IPD harmonized dataset and summary.")

Saved RESOLVE-IPD harmonized dataset and summary.


In [6]:
from src.causal.estimands import pooled_survival_difference

pooled_resolve = pooled_survival_difference(df_resolve, t0=12.0)
pooled_resolve

,t0,S0,S1,Delta
0,12.0,0.267098,0.391952,0.124854


In [8]:
from pathlib import Path

DATA_DIR = Path("../data/processed")

naive_pooled = pd.read_csv(DATA_DIR / "naive_pooled_effect.csv")

naive_pooled

,t0,S0,S1,Delta
0,12.0,0.280376,0.449162,0.168785


In [9]:
comparison_naive = pd.DataFrame({
    "method": ["KM-GPT", "RESOLVE-IPD"],
    "Delta": [
        naive_pooled["Delta"].iloc[0],
        pooled_resolve["Delta"].iloc[0],
    ],
})

comparison_naive

,method,Delta
0,KM-GPT,0.168785
1,RESOLVE-IPD,0.124854


## Robustness to Reconstruction Method: Naive Estimates

The naive pooled survival contrast at $t_0 = 12$ differs across reconstruction methods:

- KM-GPT: $\Delta \approx 0.169$
- RESOLVE-IPD: $\Delta \approx 0.125$

Both methods yield a positive treatment effect, indicating a consistent survival benefit of immunotherapy. However, the magnitude of the effect differs noticeably, with KM-GPT producing a larger estimate.

This suggests that naive pooled estimates are sensitive to the reconstruction procedure, highlighting the importance of downstream adjustment and uncertainty quantification.

Because df_resolve is overall-trial data and has no subgroup column, we need a trial-level auxiliary table. For robustness, use one row per trial by collapsing combined_aux.

In [10]:
combined_aux = pd.read_csv("../data/processed/combined_auxiliary_summaries.csv")
aux_resolve = pd.DataFrame({
    "trial_id": ["KEYNOTE-181", "ESCORT", "ATTRACTION-3"],
    "age_median": [63.0, 60.0, 65.0],
    "male_rate": [0.869, 0.91, 0.85],
    "ecog0_rate": [0.401, 0.20, 0.48],
    "metastatic_rate": [0.924, 0.58, 0.88],
    "biomarker_rate": [0.70, 0.85, 0.50],   # 0.50 as midpoint of ATTRACTION-3 low/high
})

df_resolve_aug = df_resolve.merge(
    aux_resolve,
    on="trial_id",
    how="left",
)

print(df_resolve_aug.shape)
df_resolve_aug.head()

(1268, 10)


,trial_id,time,event,arm_label,treatment,age_median,male_rate,ecog0_rate,metastatic_rate,biomarker_rate
0,ATTRACTION-3,0.631283,1,Chemotherapy,0,65.0,0.85,0.48,0.88,0.5
1,ATTRACTION-3,0.797091,1,Chemotherapy,0,65.0,0.85,0.48,0.88,0.5
2,ATTRACTION-3,1.163915,1,Chemotherapy,0,65.0,0.85,0.48,0.88,0.5
3,ATTRACTION-3,1.344771,1,Chemotherapy,0,65.0,0.85,0.48,0.88,0.5
4,ATTRACTION-3,1.621011,1,Chemotherapy,0,65.0,0.85,0.48,0.88,0.5


In [11]:
# Simulate covariates for RESOLVE IPD
from src.covariates.augmentation import (
    simulate_covariates_from_aux,
    summarize_simulated_covariates,
)

df_resolve_cov = simulate_covariates_from_aux(
    df_resolve_aug,
    age_sd=8.0,
    stage_sd=0.25,
    random_state=733,
)

resolve_cov_summary = summarize_simulated_covariates(
    df_resolve_cov.assign(subgroup="overall")
)

print(df_resolve_cov.shape)
resolve_cov_summary

(1268, 16)


,trial_id,subgroup,age_mean,male_rate,ecog0_rate,metastatic_rate,biomarker_rate,stage_mean,n
0,ATTRACTION-3,overall,65.074142,0.847255,0.501193,0.878282,0.474940,2.182997,419
1,ESCORT,overall,59.462365,0.912946,0.200893,0.580357,0.868304,2.171476,448
2,KEYNOTE-181,overall,63.696029,0.887781,0.384040,0.945137,0.728180,2.358846,401


In [12]:
# Compute target population weights for RESOLVE IPD
from src.causal.pooling import (
    compute_covariate_means,
    reweight_to_target,
    weighted_survival_difference,
)

covariates = ["age", "male", "ecog0", "metastatic", "biomarker", "stage"]
t0 = 12.0

target_means = pd.Series({
    "age": 60.0,
    "male": 0.85,
    "ecog0": 0.30,
    "metastatic": 0.75,
    "biomarker": 0.60,
    "stage": 2.0,
})

w_resolve = reweight_to_target(
    df=df_resolve_cov,
    covariates=covariates,
    target_means=target_means,
)

w_norm = w_resolve / w_resolve.sum()
ess_resolve = 1.0 / np.sum(w_norm**2)

print("Min weight:", w_resolve.min())
print("Max weight:", w_resolve.max())
print("ESS:", ess_resolve)

Min weight: 2.38354953381032e-05
Max weight: 0.005591251207603705
ESS: 621.1445490345683


/Users/jonathanma/Desktop/MSEnotes/733_NPBayes/FinalProject/NPBS-project-MaZhu/src/causal/pooling.py:129: RuntimeWarning: overflow encountered in matmul
  logits = X @ theta #softmax stabilization


In [13]:
weighted_means_resolve = compute_covariate_means(
    df_resolve_cov,
    covariates,
    weights=w_resolve,
)

comparison_resolve = pd.DataFrame({
    "target": target_means,
    "weighted": weighted_means_resolve,
})

comparison_resolve

,target,weighted
age,60.00,60.000547
male,0.85,0.849996
ecog0,0.30,0.300199
metastatic,0.75,0.749722
biomarker,0.60,0.599911
stage,2.00,2.000374


In [14]:
# Weighted survival effect, RESOLVE IPD
weighted_resolve = weighted_survival_difference(
    df=df_resolve_cov,
    weights=w_resolve,
    t0=t0,
    random_state=733,
)

weighted_resolve

{'t0': 12.0,
 'S0': 0.27786435282933336,
 'S1': 0.3712541222836215,
 'Delta': 0.09338976945428812}

In [15]:
# BNP Posterior, RESOLVE IPD
rng = np.random.default_rng(733)

alpha = 200
n_draws = 300

delta_draws_resolve = []

for i in range(n_draws):
    w_draw = rng.dirichlet(alpha * w_resolve)

    est = weighted_survival_difference(
        df=df_resolve_cov,
        weights=w_draw,
        t0=t0,
        random_state=733 + i,
    )

    delta_draws_resolve.append(est["Delta"])

delta_draws_resolve = np.array(delta_draws_resolve)

summary_resolve = {
    "mean": float(delta_draws_resolve.mean()),
    "sd": float(delta_draws_resolve.std()),
    "q025": float(np.quantile(delta_draws_resolve, 0.025)),
    "q975": float(np.quantile(delta_draws_resolve, 0.975)),
}

summary_resolve

{'mean': 0.12059127975508738,
 'sd': 0.07119161768379784,
 'q025': -0.016622290817267092,
 'q975': 0.2521686897138819}

## Comparison to KMGPT

In [16]:
weighted_kmgpt = pd.read_csv("../data/processed/weighted_estimate.csv")
bnp_kmgpt = pd.read_csv("../data/processed/bnp_summary.csv")
naive_kmgpt = pd.read_csv("../data/processed/naive_pooled_effect.csv")

In [17]:
robustness_compare = pd.DataFrame({
    "method": ["KM-GPT", "RESOLVE-IPD"],
    "naive_delta": [
        naive_kmgpt["Delta"].iloc[0],
        pooled_resolve["Delta"].iloc[0],
    ],
    "weighted_delta": [
        weighted_kmgpt["Delta"].iloc[0],
        weighted_resolve["Delta"],
    ],
    "bnp_mean": [
        bnp_kmgpt["mean"].iloc[0],
        summary_resolve["mean"],
    ],
    "bnp_q025": [
        bnp_kmgpt["q025"].iloc[0],
        summary_resolve["q025"],
    ],
    "bnp_q975": [
        bnp_kmgpt["q975"].iloc[0],
        summary_resolve["q975"],
    ],
})

robustness_compare

,method,naive_delta,weighted_delta,bnp_mean,bnp_q025,bnp_q975
0,KM-GPT,0.168785,0.234112,0.205048,0.051695,0.360661
1,RESOLVE-IPD,0.124854,0.093390,0.120591,-0.016622,0.252169


## Robustness to Reconstruction Method

We compared results obtained using KM-GPT and RESOLVE-IPD reconstructions under the same downstream causal framework.

### Naive pooled estimates
- KM-GPT: $\Delta \approx 0.169$
- RESOLVE-IPD: $\Delta \approx 0.125$

Naive estimates differ across reconstruction methods, indicating sensitivity of unadjusted analyses to the reconstruction procedure.

### Target-population weighted estimates
- KM-GPT: $\Delta \approx 0.234$
- RESOLVE-IPD: $\Delta \approx 0.093$

After reweighting, the estimated treatment effect remains positive under both methods but differs in magnitude. This suggests that both reconstruction and population definition influence the estimated effect size.

### Bayesian nonparametric uncertainty
- KM-GPT: mean $\approx 0.205$, 95% CI $(0.052, 0.361)$
- RESOLVE-IPD: mean $\approx 0.121$, 95% CI $(-0.017, 0.252)$

The RESOLVE-IPD-based posterior exhibits greater uncertainty and includes zero, indicating weaker evidence for a treatment effect under this reconstruction.

### Interpretation

While both pipelines agree on the direction of the treatment effect in point estimates, the magnitude and statistical uncertainty differ meaningfully across reconstruction methods. This highlights the importance of accounting for upstream reconstruction uncertainty when performing causal inference from reconstructed survival data.

In [19]:
from pathlib import Path
import pandas as pd

output_dir = Path("../data/processed")
output_dir.mkdir(parents=True, exist_ok=True)

# 1. Harmonized RESOLVE dataset
df_resolve.to_csv(output_dir / "resolve_ipd_harmonized.csv", index=False)

# 2. RESOLVE summary table
resolve_summary.to_csv(output_dir / "resolve_ipd_summary.csv", index=False)

# 3. Naive RESOLVE estimate
pooled_resolve.to_csv(
    output_dir / "resolve_naive_pooled_effect.csv",
    index=False
)

# 4. RESOLVE weights
pd.DataFrame({
    "weight": w_resolve
}).to_csv(output_dir / "resolve_target_weights.csv", index=False)

# 5. Weight diagnostics
pd.DataFrame({
    "ESS": [ess_resolve],
    "min_weight": [w_resolve.min()],
    "max_weight": [w_resolve.max()],
}).to_csv(output_dir / "resolve_weight_diagnostics.csv", index=False)

# 6. Target vs weighted covariates
comparison_resolve.to_csv(
    output_dir / "resolve_target_vs_weighted_covariates.csv",
    index=False
)

# 7. Weighted estimate
pd.DataFrame([weighted_resolve]).to_csv(
    output_dir / "resolve_weighted_estimate.csv",
    index=False
)

# 8. BNP posterior draws
pd.DataFrame({
    "delta": delta_draws_resolve
}).to_csv(output_dir / "resolve_bnp_delta_draws.csv", index=False)

# 9. BNP summary
pd.DataFrame([summary_resolve]).to_csv(
    output_dir / "resolve_bnp_summary.csv",
    index=False
)

# 10. Final comparison table
robustness_compare.to_csv(
    output_dir / "reconstruction_robustness_comparison.csv",
    index=False
)

print("Saved all RESOLVE-IPD robustness outputs.")

Saved all RESOLVE-IPD robustness outputs.
